# Complete Data Cleaning & Exploratory Data Analysis (EDA)
### Loan Eligibility • Adult Census Income • Hotel Booking

**Libraries:** Pandas, NumPy, Matplotlib and Seaborn

**Instructions:** Keep the three CSV files in the same folder as this notebook and run every cell from top to bottom. The notebook analyses every dataset column, creates suitable visualizations, handles missing values and duplicates, and saves cleaned CSV files.

**Privacy note:** The Hotel Booking file contains direct personal/contact fields (`name`, `email`, `phone-number`, `credit_card`). The notebook reports their column-level structure and missingness, but does not display their values or plot them. These fields are excluded from the privacy-safe cleaned export.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

---
# Dataset 1 — Loan Eligibility

**Source file:** `loan-train.csv`

Load the dataset and inspect its dimensions, columns, data types, sample records, descriptive statistics, missing values and duplicates.

In [ ]:
loan = pd.read_csv("loan-train.csv")
print("Dataset shape:", loan.shape)
print("\nColumn names:")
print(loan.columns.tolist())
display(loan.head())
print("\nDataset information:")
loan.info()
print("\nDescriptive statistics (all columns):")
display(loan.describe(include="all").T)
print("\nMissing values:")
display(loan.isnull().sum().to_frame("Missing Values"))
print("\nDuplicate rows:", loan.duplicated().sum())

In [ ]:
# Column-by-column analysis and suitable visualizations.
# Numeric columns: distribution histogram and boxplot.
# Categorical columns: frequency table and top-category count plot.

In [ ]:
for col in loan.columns:
    print("\n" + "="*70)
    print("COLUMN:", col)
    print("Data type:", loan[col].dtype)
    print("Missing values:", loan[col].isna().sum())
    print("Unique values:", loan[col].nunique(dropna=True))
    if pd.api.types.is_numeric_dtype(loan[col]):
        display(loan[col].describe().to_frame("Statistics"))
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(loan[col].dropna(), kde=True, ax=axes[0])
        axes[0].set_title(f"Distribution of {col}")
        sns.boxplot(x=loan[col], ax=axes[1])
        axes[1].set_title(f"Boxplot of {col}")
        plt.tight_layout()
        plt.show()
    else:
        display(loan[col].value_counts(dropna=False).head(20).to_frame("Frequency"))
        counts = loan[col].fillna("Missing").astype(str).value_counts().head(15)
        plt.figure(figsize=(10, 4))
        sns.barplot(x=counts.values, y=counts.index)
        plt.title(f"Top categories in {col}")
        plt.xlabel("Count")
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()

In [ ]:
# Data cleaning
loan_clean = loan.copy()
before_rows = len(loan_clean)
loan_clean = loan_clean.drop_duplicates().copy()
print("Duplicate rows removed:", before_rows - len(loan_clean))

# Standardize whitespace in text columns.
for col in loan_clean.select_dtypes(include=["object"]).columns:
    loan_clean[col] = loan_clean[col].apply(
        lambda value: value.strip() if isinstance(value, str) else value
    )

# Loan dataset: Dependents may contain the text value '3+'.
if "Dependents" in loan_clean.columns:
    loan_clean["Dependents"] = loan_clean["Dependents"].replace("3+", "3")
    loan_clean["Dependents"] = pd.to_numeric(loan_clean["Dependents"], errors="coerce")

# Impute missing numeric values with median and categorical values with mode.
for col in loan_clean.columns:
    if loan_clean[col].isna().any():
        if pd.api.types.is_numeric_dtype(loan_clean[col]):
            median_value = loan_clean[col].median()
            if pd.notna(median_value):
                loan_clean[col] = loan_clean[col].fillna(median_value)
        else:
            mode_value = loan_clean[col].mode(dropna=True)
            fill_value = mode_value.iloc[0] if not mode_value.empty else "Unknown"
            loan_clean[col] = loan_clean[col].fillna(fill_value)

print("Cleaned shape:", loan_clean.shape)
print("\nRemaining missing values:")
display(loan_clean.isnull().sum().to_frame("Missing Values"))
print("\nRemaining duplicate rows:", loan_clean.duplicated().sum())
loan_clean.to_csv("loan_cleaned.csv", index=False)
print("Saved: loan_cleaned.csv")

In [ ]:
# Target distribution and numeric correlation
if "Loan_Status" in loan_clean.columns:
    plt.figure(figsize=(6, 4))
    sns.countplot(data=loan_clean, x="Loan_Status")
    plt.title("Loan Status Distribution")
    plt.tight_layout()
    plt.show()
numeric_loan = loan_clean.select_dtypes(include=np.number)
if numeric_loan.shape[1] > 1:
    plt.figure(figsize=(10, 7))
    sns.heatmap(numeric_loan.corr(), annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Loan Dataset Numeric Correlation Heatmap")
    plt.tight_layout()
    plt.show()

---
# Dataset 2 — Adult Census Income

**Source file:** `adult.csv`

Load the dataset and inspect its dimensions, columns, data types, sample records, descriptive statistics, missing values and duplicates.

In [ ]:
adult = pd.read_csv("adult.csv")
print("Dataset shape:", adult.shape)
print("\nColumn names:")
print(adult.columns.tolist())
display(adult.head())
print("\nDataset information:")
adult.info()
print("\nDescriptive statistics (all columns):")
display(adult.describe(include="all").T)
print("\nMissing values:")
display(adult.isnull().sum().to_frame("Missing Values"))
print("\nDuplicate rows:", adult.duplicated().sum())

In [ ]:
# Column-by-column analysis and suitable visualizations.
# Numeric columns: distribution histogram and boxplot.
# Categorical columns: frequency table and top-category count plot.

In [ ]:
for col in adult.columns:
    print("\n" + "="*70)
    print("COLUMN:", col)
    print("Data type:", adult[col].dtype)
    print("Missing values:", adult[col].isna().sum())
    print("Unique values:", adult[col].nunique(dropna=True))
    if pd.api.types.is_numeric_dtype(adult[col]):
        display(adult[col].describe().to_frame("Statistics"))
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(adult[col].dropna(), kde=True, ax=axes[0])
        axes[0].set_title(f"Distribution of {col}")
        sns.boxplot(x=adult[col], ax=axes[1])
        axes[1].set_title(f"Boxplot of {col}")
        plt.tight_layout()
        plt.show()
    else:
        display(adult[col].value_counts(dropna=False).head(20).to_frame("Frequency"))
        counts = adult[col].fillna("Missing").astype(str).value_counts().head(15)
        plt.figure(figsize=(10, 4))
        sns.barplot(x=counts.values, y=counts.index)
        plt.title(f"Top categories in {col}")
        plt.xlabel("Count")
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()

In [ ]:
# Data cleaning
adult_clean = adult.copy()
before_rows = len(adult_clean)
adult_clean = adult_clean.drop_duplicates().copy()
print("Duplicate rows removed:", before_rows - len(adult_clean))

# Standardize whitespace in text columns.
for col in adult_clean.select_dtypes(include=["object"]).columns:
    adult_clean[col] = adult_clean[col].apply(
        lambda value: value.strip() if isinstance(value, str) else value
    )

# Loan dataset: Dependents may contain the text value '3+'.
if "Dependents" in adult_clean.columns:
    adult_clean["Dependents"] = adult_clean["Dependents"].replace("3+", "3")
    adult_clean["Dependents"] = pd.to_numeric(adult_clean["Dependents"], errors="coerce")

# Impute missing numeric values with median and categorical values with mode.
for col in adult_clean.columns:
    if adult_clean[col].isna().any():
        if pd.api.types.is_numeric_dtype(adult_clean[col]):
            median_value = adult_clean[col].median()
            if pd.notna(median_value):
                adult_clean[col] = adult_clean[col].fillna(median_value)
        else:
            mode_value = adult_clean[col].mode(dropna=True)
            fill_value = mode_value.iloc[0] if not mode_value.empty else "Unknown"
            adult_clean[col] = adult_clean[col].fillna(fill_value)

print("Cleaned shape:", adult_clean.shape)
print("\nRemaining missing values:")
display(adult_clean.isnull().sum().to_frame("Missing Values"))
print("\nRemaining duplicate rows:", adult_clean.duplicated().sum())
adult_clean.to_csv("adult_cleaned.csv", index=False)
print("Saved: adult_cleaned.csv")

In [ ]:
# Income distribution and numeric correlation
if "income" in adult_clean.columns:
    plt.figure(figsize=(6, 4))
    sns.countplot(data=adult_clean, x="income")
    plt.title("Income Class Distribution")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
numeric_adult = adult_clean.select_dtypes(include=np.number)
if numeric_adult.shape[1] > 1:
    plt.figure(figsize=(12, 9))
    sns.heatmap(numeric_adult.corr(), cmap="coolwarm", center=0)
    plt.title("Adult Dataset Numeric Correlation Heatmap")
    plt.tight_layout()
    plt.show()

---
# Dataset 3 — Hotel Booking

**Source file:** `hotel_booking.csv`

Load the dataset and inspect its dimensions, columns, data types, sample records, descriptive statistics, missing values and duplicates.

In [ ]:
hotel = pd.read_csv("hotel_booking.csv")
print("Dataset shape:", hotel.shape)
print("\nColumn names:")
print(hotel.columns.tolist())
display(hotel.head())
print("\nDataset information:")
hotel.info()
print("\nDescriptive statistics (all columns):")
display(hotel.describe(include="all").T)
print("\nMissing values:")
display(hotel.isnull().sum().to_frame("Missing Values"))
print("\nDuplicate rows:", hotel.duplicated().sum())

In [ ]:
# Keep a working copy. Sensitive direct identifiers are not displayed or visualized.
sensitive_cols = [c for c in ["name", "email", "phone-number", "credit_card"] if c in hotel.columns]
print("Sensitive columns present (values hidden):", sensitive_cols)
print("Sensitive-column missing-value counts:")
display(hotel[sensitive_cols].isna().sum().to_frame("Missing Values") if sensitive_cols else pd.DataFrame())

In [ ]:
# Column-by-column analysis and suitable visualizations.
# Numeric columns: distribution histogram and boxplot.
# Categorical columns: frequency table and top-category count plot.

In [ ]:
sensitive_cols = [c for c in ["name", "email", "phone-number", "credit_card"] if c in hotel.columns]
for col in hotel.columns:
    print("\n" + "="*70)
    print("COLUMN:", col)
    print("Data type:", hotel[col].dtype)
    print("Missing values:", hotel[col].isna().sum())
    print("Unique values:", hotel[col].nunique(dropna=True))
    if col in sensitive_cols:
        print("Sensitive field: values intentionally hidden for privacy.")
        continue
    if pd.api.types.is_numeric_dtype(hotel[col]):
        display(hotel[col].describe().to_frame("Statistics"))
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(hotel[col].dropna(), kde=True, ax=axes[0])
        axes[0].set_title(f"Distribution of {col}")
        sns.boxplot(x=hotel[col], ax=axes[1])
        axes[1].set_title(f"Boxplot of {col}")
        plt.tight_layout()
        plt.show()
    else:
        display(hotel[col].value_counts(dropna=False).head(20).to_frame("Frequency"))
        counts = hotel[col].fillna("Missing").astype(str).value_counts().head(15)
        plt.figure(figsize=(10, 4))
        sns.barplot(x=counts.values, y=counts.index)
        plt.title(f"Top categories in {col}")
        plt.xlabel("Count")
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()

In [ ]:
# Data cleaning
hotel_clean = hotel.copy()
before_rows = len(hotel_clean)
hotel_clean = hotel_clean.drop_duplicates().copy()
print("Duplicate rows removed:", before_rows - len(hotel_clean))

# Convert reservation status date to datetime where possible.
if "reservation_status_date" in hotel_clean.columns:
    hotel_clean["reservation_status_date"] = pd.to_datetime(
        hotel_clean["reservation_status_date"], errors="coerce"
    )

# Missing values: numeric columns -> median; categorical columns -> mode,
# or 'Unknown' when no mode exists.
for col in hotel_clean.columns:
    if col in ["name", "email", "phone-number", "credit_card"]:
        continue
    if hotel_clean[col].isna().any():
        if pd.api.types.is_numeric_dtype(hotel_clean[col]):
            median_value = hotel_clean[col].median()
            if pd.notna(median_value):
                hotel_clean[col] = hotel_clean[col].fillna(median_value)
        else:
            mode_value = hotel_clean[col].mode(dropna=True)
            fill_value = mode_value.iloc[0] if not mode_value.empty else "Unknown"
            hotel_clean[col] = hotel_clean[col].fillna(fill_value)

# Exclude direct personal/contact identifiers from the cleaned shareable file.
hotel_clean = hotel_clean.drop(columns=sensitive_cols, errors="ignore")
print("Cleaned shape:", hotel_clean.shape)
print("\nRemaining missing values:")
display(hotel_clean.isnull().sum().to_frame("Missing Values"))
print("\nRemaining duplicate rows:", hotel_clean.duplicated().sum())
hotel_clean.to_csv("hotel_booking_cleaned.csv", index=False)
print("Saved: hotel_booking_cleaned.csv")

In [ ]:
# Booking cancellation distribution and numeric correlation
if "is_canceled" in hotel_clean.columns:
    plt.figure(figsize=(6, 4))
    sns.countplot(data=hotel_clean, x="is_canceled")
    plt.title("Hotel Booking Cancellation Distribution (0 = No, 1 = Yes)")
    plt.tight_layout()
    plt.show()
numeric_hotel = hotel_clean.select_dtypes(include=np.number)
if numeric_hotel.shape[1] > 1:
    plt.figure(figsize=(14, 10))
    sns.heatmap(numeric_hotel.corr(), cmap="coolwarm", center=0)
    plt.title("Hotel Booking Numeric Correlation Heatmap")
    plt.tight_layout()
    plt.show()

---
# Final Summary

The notebook has:
- Loaded all three datasets in the required order.
- Inspected every column and reported its data type, missing-value count and unique-value count.
- Produced numeric distributions/boxplots and categorical frequency plots for every non-sensitive column.
- Removed duplicate rows and handled missing values.
- Saved the cleaned datasets as `loan_cleaned.csv`, `adult_cleaned.csv`, and `hotel_booking_cleaned.csv`.

**Important:** Median/mode imputation is a general-purpose cleaning choice. For a formal report, review the column-level results and explain why the chosen treatment is suitable. Outliers are visualized but are not automatically deleted because unusual values can be genuine observations.